In [0]:
import pandas as pd

# Read in Rebecca's patient data for reference
rebecca_redcap_full_path = "/Workspace/Users/user/user/UMD Data for HPO Phecode mapping.csv"

rebecca_redcap = pd.read_csv(rebecca_redcap_full_path)
print('Number of rows:', len(rebecca_redcap))
print('Number of columns:', len(rebecca_redcap.columns))
display(rebecca_redcap)

In [0]:
import pandas as pd

# Read in scrambled patient data
data_labels_full_path = "/Workspace/Users/user/user/UndiagnosedMendelian-EHRExtraction_DATA_LABELS_2026-05-05_1558.csv"

data_labels = pd.read_csv(data_labels_full_path)
print('Number of rows:', len(data_labels))
print('Number of columns:', len(data_labels.columns))
display(data_labels)

In [0]:
data_labels.columns[:100]

In [0]:
import pandas as pd

# Read in column name to hpo code transcript
hposetta_stone_full_path = "/Workspace/Users/user/user/UMD Phenotype Terms v4_final.csv"

hposetta_stone = pd.read_csv(hposetta_stone_full_path)
print('Number of rows:', len(hposetta_stone))
print('Number of columns:', len(hposetta_stone.columns))
display(hposetta_stone)

In [0]:
ignore_or_not = hposetta_stone['Ignore or not'].value_counts()
display(ignore_or_not)

In [0]:
# For all data_labels columns that match a value in hposetta_stone's "HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11" column, any with value = "ignore" in column "Ignore or not" in dataframe hposetta_stone to be removed from data_labels (Except for column Record_ID #:). Also, any data_labels column names that do not match a value for hposetta_stone's "HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11" column to be called out.

# Helper: normalize by stripping a trailing ')' and lowering case
def normalize(s):
    return s.rstrip(')').lower()

# Get columns to ignore based on hposetta_stone (account for errant spaces)
ignore_cols = hposetta_stone[
    hposetta_stone['Ignore or not'].str.strip().str.lower() == 'ignore'
]['HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11'].dropna().tolist()

# Build a normalized lookup for ignore_cols
ignore_cols_norm = {normalize(col): col for col in ignore_cols}

# Find which of those exist in data_labels (excluding Record_ID #:), using normalized comparison
cols_to_drop = [col for col in data_labels.columns
                if normalize(col) in ignore_cols_norm and col != 'Record ID #:']
print(f"Columns to drop (marked ignore): {len(cols_to_drop)}")
display(cols_to_drop)

# Drop them
data_labels_filtered = data_labels.drop(columns=cols_to_drop).copy()

# Find data_labels columns that don't match any value in hposetta_stone (normalized)
hpo_values = set(hposetta_stone['HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11'].dropna())
hpo_values_norm = {normalize(val): val for val in hpo_values}

labels_no_match = [col for col in data_labels_filtered.columns if normalize(col) not in hpo_values_norm]
print(f"\ndata_labels columns with no match in hposetta_stone ({len(labels_no_match)}):")
display(labels_no_match)

# hposetta_stone values with no matching column in data_labels
data_labels_norm = {normalize(col): col for col in data_labels.columns}
hpo_no_match = [val for val in hpo_values if normalize(val) not in data_labels_norm]
print(f"\nhposetta_stone values with no column in data_labels ({len(hpo_no_match)}):")
display(hpo_no_match)

In [0]:
# Rename column in data_labels so it matches hposetta_stone
data_labels = data_labels.rename(columns={
    'Other testing: (choice=Array Comparative Genomic Hybridization (CGH or CMA))':
    'Other testing: (choice=Array Comparative Genomic Hybridization (CGH))'
})

# Helper: normalize by stripping a trailing ')' and lowering case
def normalize(s):
    return s.rstrip(')').lower()

# Get columns to ignore based on hposetta_stone (account for errant spaces)
ignore_cols = hposetta_stone[
    hposetta_stone['Ignore or not'].str.strip().str.lower() == 'ignore'
]['HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11'].dropna().tolist()

# Build a normalized lookup for ignore_cols
ignore_cols_norm = {normalize(col): col for col in ignore_cols}

# Find which of those exist in data_labels (excluding Record_ID #:), using normalized comparison
cols_to_drop = [col for col in data_labels.columns
                if normalize(col) in ignore_cols_norm and col != 'Record ID #:']
print(f"Columns to drop (marked ignore): {len(cols_to_drop)}")
display(cols_to_drop)

# Drop them
data_labels_filtered = data_labels.drop(columns=cols_to_drop).copy()

# Find data_labels columns that don't match any value in hposetta_stone (normalized)
hpo_values = set(hposetta_stone['HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11'].dropna())
hpo_values_norm = {normalize(val): val for val in hpo_values}

labels_no_match = [col for col in data_labels_filtered.columns if normalize(col) not in hpo_values_norm]
print(f"\ndata_labels columns with no match in hposetta_stone ({len(labels_no_match)}):")
display(labels_no_match)

# hposetta_stone values with no matching column in data_labels
data_labels_norm = {normalize(col): col for col in data_labels.columns}
hpo_no_match = [val for val in hpo_values if normalize(val) not in data_labels_norm]
print(f"\nhposetta_stone values with no column in data_labels ({len(hpo_no_match)}):")
display(hpo_no_match)

In [0]:
# Drop all columns that had no match in hposetta_stone except brain imaging columns from labels_no_match that we want to keep
retain_cols = [
    'Brain malformations and abnormal imaging: (choice=No MRI found)',
    'Brain malformations and abnormal imaging: (choice=Normal Brain MRI)',
    'Brain malformations and abnormal imaging: (choice=Leukodystrophy)',
    'Brain malformations and abnormal imaging: (choice=Ventriculomegaly)',
    'Brain malformations and abnormal imaging: (choice=Other abnormality of corpus callosum)',
    'Brain malformations and abnormal imaging: (choice=Chiari malformation)',
]

# Remove the retained columns from the drop list
labels_to_drop = [col for col in labels_no_match if col not in retain_cols]

# Drop all unmatched columns except the ones we want to keep
data_labels_filtered2 = data_labels_filtered.drop(columns=labels_to_drop).copy()

print(f"Columns dropped: {len(labels_to_drop)}")
print(f"Columns retained from no-match list: {len([c for c in retain_cols if c in data_labels_filtered.columns])}")
print(f"Remaining columns: {len(data_labels_filtered2.columns)}")

In [0]:
# Rename data_labels columns using the "Term from checklist" mapping from hposetta_stone
rename_map = {}        # {original data_labels col name: new "Term from checklist" name}
null_checklist = []    # columns where "Term from checklist" is null

for _, row in hposetta_stone.iterrows():
    hpo_val = row['HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11']
    checklist_val = row['Term from checklist']

    if pd.isna(hpo_val):
        continue

    norm_hpo = normalize(hpo_val)

    # Check each data_labels column for a normalized match
    for col in data_labels_filtered2.columns:
        if normalize(col) == norm_hpo:
            if pd.isna(checklist_val):
                null_checklist.append(col)
            else:
                rename_map[col] = checklist_val

# Report columns with null "Term from checklist" values
if null_checklist:
    print(f"Columns matched in hposetta_stone but 'Term from checklist' is null ({len(null_checklist)}):")
    display(null_checklist)
else:
    print("All matched columns have a valid 'Term from checklist' value.")

# Report columns that were NOT renamed
not_renamed = [col for col in data_labels_filtered2.columns
               if col not in rename_map and col not in null_checklist]

if not_renamed:
    print(f"\nColumns NOT renamed ({len(not_renamed)}):")
    display(not_renamed)
else:
    print("\nAll columns were renamed.")

# Apply the renaming
print(f"\nColumns being renamed: {len(rename_map)}")
data_labels_filtered2 = data_labels_filtered2.rename(columns=rename_map)

print(f"\nUpdated column names ({len(data_labels_filtered2.columns)} total):")
display(data_labels_filtered2.columns.tolist())

In [0]:
# Pull the row where the first column matches the specified value
first_col = hposetta_stone.columns[0]
result = hposetta_stone[hposetta_stone[first_col] == 'Endocrine: (choice=Pheochromocytoma or paraganglioma)']
display(result)

In [0]:
#count of duplicate column names in data_labels_filtered2
dup_cols = data_labels_filtered2.columns[data_labels_filtered2.columns.duplicated()]
print(f"Duplicate column names: {len(dup_cols)}")
if len(dup_cols) > 0:
    display(dup_cols.tolist())

In [0]:
# Check if column Pheochromocytoma or paraganglioma exists in data_labels_filtered2
'Pheochromocytoma or paraganglioma' in data_labels_filtered2.columns

Pheochromocytoma or paraganglioma exists in data_labels_filtered2 and is not duplicated so we should be fine. 

In [0]:
print(f"\nColumns NOT renamed ({len(not_renamed)}):")
display(not_renamed)

Brain malformations and abnormal imaging: (choice=No MRI found) 
- no HPO code for this that I could find


Brain malformations and abnormal imaging: (choice=Normal Brain MRI) 
- no HPO code for this that I could find


Brain malformations and abnormal imaging: (choice=Leukodystrophy) 
- HP:002415 - https://hpo.jax.org/browse/search?q=leukodystrophy&navFilter=all



Brain malformations and abnormal imaging: (choice=Ventriculomegaly)
- HP:0002119? - https://hpo.jax.org/browse/search?q=ventriculomegaly&navFilter=all


Brain malformations and abnormal imaging: (choice=Other abnormality of corpus callosum)
https://hpo.jax.org/browse/search?q=other%20abnormality%20of%20corpus%20callosum&navFilter=all

- HP:0001273 is closest I could find. Also shown as hpo code here: https://www.ncbi.nlm.nih.gov/medgen/334198


Brain malformations and abnormal imaging: (choice=Chiari malformation)
https://hpo.jax.org/browse/search?q=chiari%20malformation&navFilter=all

- HP:0002308



In [0]:
# Drop brain imaging columns with no associated HPO codes
data_labels_filtered3 = data_labels_filtered2.drop(columns=[
    'Brain malformations and abnormal imaging: (choice=No MRI found)',
    'Brain malformations and abnormal imaging: (choice=Normal Brain MRI)',
])

print(f"Remaining columns: {len(data_labels_filtered3.columns)}")
display(data_labels_filtered3)

In [0]:
all_values = data_labels_filtered3.iloc[:, 1:].stack().unique()
print(f"Unique values across all columns (except first): {len(all_values)}")
display(sorted(all_values, key=str))

In [0]:
# If checked, change to column name. If not, null. 
import numpy as np

data_labels_filtered4 = data_labels_filtered3.copy()

# Rename columns
data_labels_filtered4 = data_labels_filtered4.rename(columns={
    'Brain malformations and abnormal imaging: (choice=Leukodystrophy)': 'Leukodystrophy',
    'Brain malformations and abnormal imaging: (choice=Ventriculomegaly)': 'Ventriculomegaly',
    'Brain malformations and abnormal imaging: (choice=Chiari malformation)': 'Chiari malformation',
    'Brain malformations and abnormal imaging: (choice=Other abnormality of corpus callosum)': 'Other abnormality of corpus callosum',
})

for col in data_labels_filtered4.columns:
    mask_checked = data_labels_filtered4[col] == 'Checked'
    mask_unchecked = data_labels_filtered4[col] == 'Unchecked'
    data_labels_filtered4.loc[mask_checked, col] = col
    data_labels_filtered4.loc[mask_unchecked, col] = np.nan

print(f"Columns: {len(data_labels_filtered4.columns)}")
display(data_labels_filtered4.head())

In [0]:
# Add brain imaging rows to hposetta_stone with their HPO codes
new_rows = pd.DataFrame([
    {
        'HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11': 'Brain malformations and abnormal imaging: (choice=Leukodystrophy)',
        'Ignore or not': None,
        'Term from checklist': 'Leukodystrophy',
        'HPO term (text)': 'Leukodystrophy',
        'HPO term (ID)': 'HP:0002415',
    },
    {
        'HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11': 'Brain malformations and abnormal imaging: (choice=Ventriculomegaly)',
        'Ignore or not': None,
        'Term from checklist': 'Ventriculomegaly',
        'HPO term (text)': 'Ventriculomegaly',
        'HPO term (ID)': 'HP:0002119',
    },
    {
        'HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11': 'Brain malformations and abnormal imaging: (choice=Other abnormality of corpus callosum)',
        'Ignore or not': None,
        'Term from checklist': 'Other abnormality of corpus callosum',
        'HPO term (text)': 'Abnormal corpus callosum morphology',
        'HPO term (ID)': 'HP:0001273',
    },
    {
        'HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11': 'Brain malformations and abnormal imaging: (choice=Chiari malformation)',
        'Ignore or not': None,
        'Term from checklist': 'Chiari malformation',
        'HPO term (text)': 'Chiari malformation',
        'HPO term (ID)': 'HP:0002308',
    },
])

hposetta_stone2 = pd.concat([hposetta_stone, new_rows], ignore_index=True)

print(f"hposetta_stone had {len(hposetta_stone)} rows")
print(f"hposetta_stone2 has {len(hposetta_stone2)} rows")
display(hposetta_stone2.tail(4))

display(hposetta_stone2)

In [0]:
output_folder = "/Workspace/Users/user/user/"

hposetta_stone2.to_csv(output_folder + "hposetta_stone2.csv", index=False)
data_labels_filtered4.to_csv(output_folder + "data_labels_filtered4.csv", index=False)

print("Exported both files.")

In [0]:
# Quality check: Were these brain imaging conditions backfilled?
# Last filled value in column 1 of rebecca_redcap
col1_rebecca = rebecca_redcap.columns[0]
last_filled = rebecca_redcap[col1_rebecca].dropna().iloc[-1]
print(f"Last filled value in rebecca_redcap column 1: {last_filled}")

# First row in data_labels_filtered4 with a non-null value in any of the brain columns
brain_cols = ['Leukodystrophy', 'Ventriculomegaly', 'Other abnormality of corpus callosum', 'Chiari malformation']
col1_labels = data_labels_filtered4.columns[0]

has_brain = data_labels_filtered4[brain_cols].notna().any(axis=1)
first_match = data_labels_filtered4.loc[has_brain, col1_labels].iloc[0]
print(f"First column 1 value with a non-null brain column: {first_match}")